In [2]:
"""
Barcode Damage Detection - Dataset Preparation (Damage-Only Ground Truth)
=========================================================================

PROBLEM WITH PREVIOUS APPROACH:
  pyzbar fails for many reasons:
    ❌ Barcode too far away (small in frame)
    ❌ Poor lighting / shadows
    ❌ Low image resolution
    ❌ Barcode at an angle
  None of these are DAMAGE — so labelling them as FAIL was wrong.

NEW APPROACH:
  PASS = clean barcode image from the dataset (unmodified, assumed intact)
  FAIL = same clean image with synthetic damage applied (scratches, blur, marks)

  This way:
    ✅ FAIL means ONLY physical damage to the barcode
    ✅ Lighting, distance, resolution differences are NOT labelled as FAIL
    ✅ Ground truth is 100% controlled and consistent
    ✅ The model learns to detect damage, not decode failure

DATASET STRUCTURE:
  barcode_dataset_with_fail/
    train/      PASS/  FAIL/
    validation/ PASS/  FAIL/
    test/       PASS/  FAIL/
    manifest.json

Run:
    pip install datasets pillow opencv-python tqdm
    python prepare_barcode_dataset.py
"""

import os, random, shutil, json
import numpy as np
import cv2
from PIL import Image
from tqdm import tqdm

# ── Config ────────────────────────────────────────────────────────────────────
SEED         = 42
N_IMAGES     = 3000      # number of source images → gives 3000 PASS + 3000 FAIL
IMG_SIZE     = (224, 224)
OUTPUT_DIR   = "barcode_dataset_with_fail"
SPLIT_RATIOS = (0.70, 0.15, 0.15)

random.seed(SEED)
np.random.seed(SEED)

# ── Image helper ──────────────────────────────────────────────────────────────

def resize(pil_img):
    return pil_img.convert("RGB").resize(IMG_SIZE, Image.LANCZOS)


def is_good_quality(pil_img,
                    min_sharpness=500,
                    min_contrast=40,
                    min_transitions=20,
                    min_edge_density=15):
    """
    Returns True only if the image contains a clear, well-defined barcode.
    Rejects images where the barcode is:
      - Too far away  (small in the frame → low sharpness + few transitions)
      - Out of focus  (blurry → low Laplacian variance)
      - Low contrast  (washed-out or dark → std dev of pixels too low)
      - Too cluttered (barcode lost in background noise)

    How each check works
    ────────────────────
    Sharpness (Laplacian variance):
      The Laplacian filter highlights rapid intensity changes (edges).
      Its variance measures how much edge content exists overall.
      A sharp barcode has strong, crisp bar edges → high variance.
      A far-away or blurry barcode has soft edges → low variance.
      Threshold: 500  (images 00086=174, 00137=178 → rejected)

    Contrast (pixel std dev):
      Measures the spread of pixel intensities across the image.
      A barcode needs clear dark bars on a light background.
      Low std dev means the image is too grey/uniform — bars are not visible.
      Threshold: 40

    Bar transitions (mid-row edge count):
      Samples the middle horizontal row of the image and counts how many
      times intensity changes by more than 30 — each change is a bar edge.
      A real barcode has many transitions (20+). A far-away barcode has
      very few because the bars are too small to resolve as distinct edges.
      Threshold: 20  (images 00086=3, 00137=6 → rejected)

    Edge density (Sobel mean):
      Mean value of the vertical Sobel filter across the whole image.
      Measures how much vertical-bar content is present globally.
      A barcode-dominated image has high vertical edge density.
      Threshold: 15
    """
    arr  = np.array(pil_img.convert("RGB"))
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)

    # Check 1: sharpness
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    if sharpness < min_sharpness:
        return False, f"too blurry (sharpness={sharpness:.0f} < {min_sharpness})"

    # Check 2: contrast
    contrast = gray.std()
    if contrast < min_contrast:
        return False, f"low contrast ({contrast:.1f} < {min_contrast})"

    # Check 3: bar transitions in the middle row
    mid_row     = gray[gray.shape[0] // 2, :]
    transitions = int(np.sum(np.abs(np.diff(mid_row.astype(int))) > 30))
    if transitions < min_transitions:
        return False, f"too few bar transitions ({transitions} < {min_transitions})"

    # Check 4: vertical edge density
    sobel        = np.uint8(np.absolute(cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)))
    edge_density = float(sobel.mean())
    if edge_density < min_edge_density:
        return False, f"low edge density ({edge_density:.1f} < {min_edge_density})"

    return True, "ok"


# These simulate real-world barcode damage:
#   - scratches / pen marks / scuff marks
#   - ink smudges or smears
#   - partial covering / sticker over barcode
#   - print quality degradation
#
# NOT included (these are NOT damage):
#   - blur (could be camera focus, not barcode damage)
#   - rotation (barcode is fine, just angled)
#   - noise (camera quality issue, not barcode damage)

def add_scratches(a):
    """Random lines across bars — simulates physical scratches or pen marks."""
    h, w = a.shape[:2]
    n_scratches = random.randint(3, 8)
    for _ in range(n_scratches):
        # Horizontal scratches are most damaging to vertical barcodes
        x1 = random.randint(0, w - 1)
        x2 = random.randint(0, w - 1)
        y  = random.randint(h // 4, 3 * h // 4)   # target the bar area
        thickness = random.randint(1, 4)
        color = random.choice([(0,0,0), (80,80,80), (200,200,200)])  # dark or light scratch
        cv2.line(a, (x1, y), (x2, y), color, thickness)
    return a

def add_diagonal_scratches(a):
    """Diagonal scratches — simulates scuff marks."""
    h, w = a.shape[:2]
    for _ in range(random.randint(2, 5)):
        x1, y1 = random.randint(0, w-1), random.randint(0, h-1)
        x2 = x1 + random.randint(-w//3, w//3)
        y2 = y1 + random.randint(-h//3, h//3)
        thickness = random.randint(1, 3)
        color = random.choice([(0,0,0), (60,60,60)])
        cv2.line(a, (x1,y1), (x2,y2), color, thickness)
    return a

def add_ink_smudge(a):
    """Dark blotch over part of the barcode — simulates ink smear or wet damage."""
    h, w = a.shape[:2]
    # Small irregular dark patch over the bars
    cx = random.randint(w // 4, 3 * w // 4)
    cy = random.randint(h // 4, 3 * h // 4)
    rw = random.randint(w // 8, w // 3)
    rh = random.randint(h // 8, h // 3)
    overlay = a.copy()
    cv2.ellipse(overlay, (cx, cy), (rw, rh), 0, 0, 360,
                (random.randint(0, 60), 0, 0), -1)
    alpha = random.uniform(0.4, 0.8)
    a = cv2.addWeighted(overlay, alpha, a, 1 - alpha, 0)
    return a

def add_partial_cover(a):
    """Rectangle covering part of bars — simulates sticker or tape over barcode."""
    h, w = a.shape[:2]
    # Cover a vertical strip of the barcode (most damaging)
    x_start = random.randint(w // 5, w // 2)
    cover_w  = random.randint(w // 8, w // 4)
    color = random.choice([
        (255, 255, 255),    # white sticker
        (255, 255, 180),    # yellow tape
        (200, 200, 200),    # grey tape
    ])
    a[0:h, x_start:x_start + cover_w] = color
    return a

def add_print_degradation(a):
    """
    Simulates poor print quality — faded or partially missing bars.
    Randomly lightens vertical strips to simulate ink running out.
    """
    h, w = a.shape[:2]
    n_fades = random.randint(3, 8)
    for _ in range(n_fades):
        x = random.randint(0, w - 1)
        fade_w = random.randint(1, 3)
        fade_amount = random.randint(80, 160)
        strip = a[0:h, x:x+fade_w].astype(np.int32)
        strip = np.clip(strip + fade_amount, 0, 255).astype(np.uint8)
        a[0:h, x:x+fade_w] = strip
    return a

def add_water_damage(a):
    """Wavy distortion + darkening — simulates moisture/water damage."""
    h, w = a.shape[:2]
    map_x = np.zeros((h, w), dtype=np.float32)
    map_y = np.zeros((h, w), dtype=np.float32)
    amplitude = random.randint(3, 8)
    frequency = random.uniform(0.05, 0.15)
    for i in range(h):
        for j in range(w):
            map_x[i, j] = j + amplitude * np.sin(2 * np.pi * i * frequency)
            map_y[i, j] = i + amplitude * np.cos(2 * np.pi * j * frequency)
    a = cv2.remap(a, map_x, map_y, cv2.INTER_LINEAR, borderValue=(255,255,255))
    dark_mask = np.random.random((h, w)) > 0.85
    a[dark_mask] = np.clip(a[dark_mask].astype(int) - random.randint(40, 80), 0, 255)
    return a.astype(np.uint8)


def add_torn_edge(a):
    """
    Simulates a torn or ripped barcode label — like images 1, 2, 3 you shared.

    Creates a jagged tear line that removes a chunk from one edge of the label.
    The torn region is filled with a background colour (white/grey) to simulate
    the label backing showing through, or just missing entirely.

    Three tear styles chosen randomly:
      - Corner tear  : diagonal rip from one corner
      - Edge tear    : jagged strip missing from left or right side
      - Bottom tear  : ragged bottom edge (label peeling off)
    """
    h, w = a.shape[:2]
    bg_color = tuple(int(c) for c in np.percentile(
        a[0:10, 0:10], 90, axis=(0, 1)).tolist())  # sample background from corner

    style = random.choice(["corner", "edge", "bottom"])

    if style == "corner":
        # Jagged diagonal tear from one corner
        corner = random.choice(["tl", "tr", "bl", "br"])
        depth_x = random.randint(w // 5, w // 2)
        depth_y = random.randint(h // 5, h // 2)
        n_points = random.randint(8, 16)

        if corner == "tr":
            xs = np.linspace(w - depth_x, w, n_points).astype(int)
            ys = np.linspace(0, depth_y, n_points).astype(int)
            # Add jitter to make it look torn not cut
            xs = np.clip(xs + np.random.randint(-8, 8, n_points), 0, w)
            ys = np.clip(ys + np.random.randint(-8, 8, n_points), 0, h)
            pts = np.array(list(zip(xs, ys)) + [(w, 0)], dtype=np.int32)
        elif corner == "tl":
            xs = np.linspace(0, depth_x, n_points).astype(int)
            ys = np.linspace(depth_y, 0, n_points).astype(int)
            xs = np.clip(xs + np.random.randint(-8, 8, n_points), 0, w)
            ys = np.clip(ys + np.random.randint(-8, 8, n_points), 0, h)
            pts = np.array([(0, 0)] + list(zip(xs, ys)), dtype=np.int32)
        elif corner == "br":
            xs = np.linspace(w, w - depth_x, n_points).astype(int)
            ys = np.linspace(h - depth_y, h, n_points).astype(int)
            xs = np.clip(xs + np.random.randint(-8, 8, n_points), 0, w)
            ys = np.clip(ys + np.random.randint(-8, 8, n_points), 0, h)
            pts = np.array(list(zip(xs, ys)) + [(w, h)], dtype=np.int32)
        else:  # bl
            xs = np.linspace(depth_x, 0, n_points).astype(int)
            ys = np.linspace(h, h - depth_y, n_points).astype(int)
            xs = np.clip(xs + np.random.randint(-8, 8, n_points), 0, w)
            ys = np.clip(ys + np.random.randint(-8, 8, n_points), 0, h)
            pts = np.array([(0, h)] + list(zip(xs, ys)), dtype=np.int32)

        cv2.fillPoly(a, [pts.reshape(-1, 1, 2)], bg_color)

    elif style == "edge":
        # Jagged strip removed from left or right edge
        side  = random.choice(["left", "right"])
        depth = random.randint(w // 6, w // 3)
        n_pts = random.randint(10, 20)
        ys    = np.linspace(0, h, n_pts).astype(int)

        if side == "right":
            xs = (w - depth + np.random.randint(-15, 15, n_pts))
            xs = np.clip(xs, w - depth - 20, w)
            pts = np.array([(w, 0)] + list(zip(xs, ys)) + [(w, h)],
                           dtype=np.int32)
        else:
            xs = (depth + np.random.randint(-15, 15, n_pts))
            xs = np.clip(xs, 0, depth + 20)
            pts = np.array([(0, 0)] + list(zip(xs, ys)) + [(0, h)],
                           dtype=np.int32)

        cv2.fillPoly(a, [pts.reshape(-1, 1, 2)], bg_color)

    else:  # bottom — label peeling
        depth = random.randint(h // 5, h // 2)
        n_pts = random.randint(10, 20)
        xs    = np.linspace(0, w, n_pts).astype(int)
        ys    = (h - depth + np.random.randint(-12, 12, n_pts))
        ys    = np.clip(ys, h - depth - 15, h)
        pts   = np.array([(0, h)] + list(zip(xs, ys)) + [(w, h)],
                         dtype=np.int32)
        cv2.fillPoly(a, [pts.reshape(-1, 1, 2)], bg_color)

    return a


def add_missing_middle(a):
    """
    Simulates a barcode with a section missing from the middle — like
    images 4 and 5 you shared (bars corrupted or absent in a band).

    Three sub-types:
      - Horizontal band  : a wide strip across the middle has bars destroyed
      - Vertical gap     : a column of bars is completely missing/white
      - Shatter pattern  : irregular blob in the centre where bars are gone
    """
    h, w = a.shape[:2]

    style = random.choice(["h_band", "v_gap", "shatter"])

    if style == "h_band":
        # Wide horizontal band across the bar zone — bars replaced with white/noise
        band_h     = random.randint(h // 6, h // 3)
        band_y     = random.randint(h // 4, h // 2)
        fill_style = random.choice(["white", "noise", "grey"])

        if fill_style == "white":
            a[band_y:band_y + band_h, :] = 255
        elif fill_style == "grey":
            a[band_y:band_y + band_h, :] = random.randint(160, 220)
        else:  # noise — like peeled print
            noise = np.random.randint(180, 255,
                                      (band_h, w, 3), dtype=np.uint8)
            a[band_y:band_y + band_h, :] = noise

    elif style == "v_gap":
        # One or two vertical columns completely missing — bars gone
        n_gaps = random.randint(1, 3)
        for _ in range(n_gaps):
            gap_x = random.randint(w // 5, 4 * w // 5)
            gap_w = random.randint(w // 8, w // 4)
            a[:, gap_x:gap_x + gap_w] = 255

    else:  # shatter — irregular centre destruction
        # Blob of white/grey where bars are shattered (like image 4)
        n_blobs = random.randint(2, 5)
        for _ in range(n_blobs):
            cx = random.randint(w // 4, 3 * w // 4)
            cy = random.randint(h // 4, 3 * h // 4)
            rw = random.randint(w // 8, w // 3)
            rh = random.randint(h // 6, h // 2)
            # Irregular polygon instead of ellipse — looks more like real damage
            n_pts   = random.randint(6, 10)
            angles  = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
            r_var   = np.random.uniform(0.6, 1.0, n_pts)
            pts_x   = (cx + rw * r_var * np.cos(angles)).astype(int)
            pts_y   = (cy + rh * r_var * np.sin(angles)).astype(int)
            pts_x   = np.clip(pts_x, 0, w - 1)
            pts_y   = np.clip(pts_y, 0, h - 1)
            pts     = np.stack([pts_x, pts_y], axis=1).reshape(-1, 1, 2)
            fill    = random.choice([255, 230, 200])
            cv2.fillPoly(a, [pts], (fill, fill, fill))

    return a


# All damage functions — each simulates a PHYSICAL defect
DAMAGE_FNS = [
    ("scratches",        add_scratches),
    ("diagonal_scratch", add_diagonal_scratches),
    ("ink_smudge",       add_ink_smudge),
    ("partial_cover",    add_partial_cover),
    ("print_degrade",    add_print_degradation),
    ("water_damage",     add_water_damage),
    ("torn_edge",        add_torn_edge),        # NEW — jagged tear from edge/corner
    ("missing_middle",   add_missing_middle),   # NEW — bars absent in a region
]

def apply_damage(pil_img):
    """Apply 1-2 random physical damage types."""
    a = np.array(pil_img.convert("RGB"))
    chosen = random.sample(DAMAGE_FNS, k=random.randint(1, 2))
    for name, fn in chosen:
        a = fn(a)
    return Image.fromarray(a.astype(np.uint8))

# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    from datasets import load_dataset

    tmp_pass = "tmp_pass"
    tmp_fail = "tmp_fail"
    for d in [tmp_pass, tmp_fail, OUTPUT_DIR]:
        os.makedirs(d, exist_ok=True)

    # ── Step 1: Stream source images ──────────────────────────────────────────
    print("=" * 60)
    print("Step 1/5 — Streaming clean barcode images...")
    print("=" * 60)
    print("  NEW STRATEGY:")
    print("  PASS = clean images from dataset (assumed physically intact)")
    print("  FAIL = same images with synthetic physical damage applied")
    print("  (No pyzbar — distance/lighting/angle are NOT treated as damage)\n")

    ds = load_dataset(
        "amaye15/barcodes-google-ocr",
        split="train",
        streaming=True,
        columns=["pixel_values"],
    ).batch(1)

    # ── Step 2: Collect clean PASS images ─────────────────────────────────────
    print("=" * 60)
    print(f"Step 2/5 — Collecting {N_IMAGES} clean source images...")
    print("=" * 60)
    print("  Quality filter ON — rejecting images where barcode is:")
    print("  far away, blurry, low contrast, or lost in background\n")

    collected  = 0
    scanned    = 0
    rejected   = 0
    reject_log = {"too blurry": 0, "low contrast": 0,
                  "too few bar transitions": 0, "low edge density": 0}

    with tqdm(total=N_IMAGES, desc="  Collecting", unit=" imgs") as pbar:
        for batch in ds:
            scanned += 1
            try:
                img = resize(batch["pixel_values"][0])
            except Exception:
                rejected += 1
                continue

            # ── Quality gate ──────────────────────────────────────────────────
            passed, reason = is_good_quality(img)
            if not passed:
                rejected += 1
                # Track which check failed most often
                for key in reject_log:
                    if key in reason:
                        reject_log[key] += 1
                        break
                del img
                continue
            # ─────────────────────────────────────────────────────────────────

            img.save(os.path.join(tmp_pass, f"pass_{collected:05d}.jpg"),
                     quality=90)
            del img
            collected += 1
            pbar.update(1)
            pbar.set_postfix(scanned=scanned, rejected=rejected)

            if collected >= N_IMAGES:
                break

    print(f"\n  Scanned   : {scanned} images from dataset")
    print(f"  Collected : {collected} passed quality filter")
    print(f"  Rejected  : {rejected} ({rejected/max(scanned,1)*100:.1f}%)")
    print(f"  Rejection reasons:")
    for reason, count in reject_log.items():
        print(f"    {reason:30s}: {count}")
    print()

    # ── Step 3: Generate FAIL images by applying physical damage ──────────────
    print("=" * 60)
    print("Step 3/5 — Generating FAIL images (physical damage simulation)...")
    print("=" * 60)
    print("  Damage types: scratches, diagonal marks, ink smudge,")
    print("                partial cover, print degradation, water damage\n")

    pass_paths = sorted([os.path.join(tmp_pass, f) for f in os.listdir(tmp_pass)])
    fail_count = 0

    with tqdm(total=N_IMAGES, desc="  Damaging", unit=" imgs") as pbar:
        for src_path in pass_paths:
            try:
                src = Image.open(src_path)
                dmg = apply_damage(src)
                src.close()
                dmg.save(os.path.join(tmp_fail, f"fail_{fail_count:05d}.jpg"), quality=90)
                dmg.close()
                fail_count += 1
                pbar.update(1)
            except Exception as e:
                print(f"  Warning: {e}")
                continue

    print(f"\n  Generated {fail_count} FAIL images\n")

    # ── Step 4: Build balanced splits ─────────────────────────────────────────
    print("=" * 60)
    print("Step 4/5 — Building 70 / 15 / 15 splits...")
    print("=" * 60)

    all_records = (
        [(f, 1) for f in sorted(os.listdir(tmp_pass))[:N_IMAGES]] +
        [(f, 0) for f in sorted(os.listdir(tmp_fail))[:N_IMAGES]]
    )
    random.shuffle(all_records)

    n       = len(all_records)
    n_train = int(n * SPLIT_RATIOS[0])
    n_val   = int(n * SPLIT_RATIOS[1])

    splits = {
        "train":      all_records[:n_train],
        "validation": all_records[n_train : n_train + n_val],
        "test":       all_records[n_train + n_val:],
    }

    # ── Step 5: Save to disk ──────────────────────────────────────────────────
    print("=" * 60)
    print("Step 5/5 — Saving dataset to disk...")
    print("=" * 60)

    manifest = {}

    for split_name, rows in splits.items():
        split_dir = os.path.join(OUTPUT_DIR, split_name)
        os.makedirs(os.path.join(split_dir, "PASS"), exist_ok=True)
        os.makedirs(os.path.join(split_dir, "FAIL"), exist_ok=True)
        n_pass = n_fail = 0
        manifest[split_name] = []

        for i, (fname, label) in enumerate(tqdm(rows, desc=f"  {split_name:12s}")):
            src_dir    = tmp_pass if label == 1 else tmp_fail
            label_name = "PASS"   if label == 1 else "FAIL"
            dst = os.path.join(split_dir, label_name, f"{i:05d}.jpg")
            shutil.copy2(os.path.join(src_dir, fname), dst)
            manifest[split_name].append({
                "file": dst, "label": label, "label_name": label_name
            })
            if label == 1: n_pass += 1
            else:          n_fail += 1

        print(f"    {split_name:12s}: {len(rows):5d} total  |  PASS={n_pass}  FAIL={n_fail}")

    with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w") as f:
        json.dump(manifest, f, indent=2)

    shutil.rmtree(tmp_pass)
    shutil.rmtree(tmp_fail)

    print(f"\n  Saved to : {os.path.abspath(OUTPUT_DIR)}\n")
    print("=" * 60)
    print("DONE! Load in PyTorch:")
    print("=" * 60)
    print("""
    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader

    T = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])

    train_ds = datasets.ImageFolder("barcode_dataset_with_fail/train",      transform=T)
    val_ds   = datasets.ImageFolder("barcode_dataset_with_fail/validation",  transform=T)
    test_ds  = datasets.ImageFolder("barcode_dataset_with_fail/test",        transform=T)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=32)
    test_loader  = DataLoader(test_ds,  batch_size=32)

    # train_ds.classes → ['FAIL', 'PASS']
    """)


if __name__ == "__main__":
    main()

Step 1/5 — Streaming clean barcode images...
  NEW STRATEGY:
  PASS = clean images from dataset (assumed physically intact)
  FAIL = same images with synthetic physical damage applied
  (No pyzbar — distance/lighting/angle are NOT treated as damage)



Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Step 2/5 — Collecting 3000 clean source images...
  Quality filter ON — rejecting images where barcode is:
  far away, blurry, low contrast, or lost in background



  Collecting: 100%|████████████████████████████████| 3000/3000 [09:05<00:00,  5.50 imgs/s, rejected=2142, scanned=5142]


...

    train       :  4200 total  |  PASS=2083  FAIL=2117


  validation  : 100%|███████████████████████████████████████████████████████████████| 900/900 [00:03<00:00, 280.00it/s]


    validation  :   900 total  |  PASS=456  FAIL=444


  test        : 100%|███████████████████████████████████████████████████████████████| 900/900 [00:03<00:00, 271.29it/s]


    test        :   900 total  |  PASS=461  FAIL=439

  Saved to : C:\Users\erum8\barcode_dataset_with_fail

DONE! Load in PyTorch:

    from torchvision import datasets, transforms
    from torch.utils.data import DataLoader

    T = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3),
    ])

    train_ds = datasets.ImageFolder("barcode_dataset_with_fail/train",      transform=T)
    val_ds   = datasets.ImageFolder("barcode_dataset_with_fail/validation",  transform=T)
    test_ds  = datasets.ImageFolder("barcode_dataset_with_fail/test",        transform=T)

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=32)
    test_loader  = DataLoader(test_ds,  batch_size=32)

    # train_ds.classes → ['FAIL', 'PASS']
    
